<a href="https://colab.research.google.com/github/Bilra-Hkr/BigData26_B_2411533007_DerielChaerahman/blob/main/Praktikum4/BD_P04_2411533007_DerielChaerahman.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# K-1. Menyiapkan Lingkungan dan Data Contoh

In [ ]:
!pip install -q duckdb networkx shapely tinydb "deltalake>=1.0"

import os, time, json, sqlite3, random, shutil, glob, zipfile
import numpy as np, pandas as pd
import pyarrow.dataset as ds
from google.colab import drive

drive.mount("/content/drive")

DIR_P3 = "/content/drive/MyDrive/BigData/Praktikum3"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum4"
ZIP_PATH = f"{DIR_P3}/trips_bersih.zip"
TRIPS_BERSIH = f"{DIR_P3}/trips_bersih"

os.makedirs(DIR_SIMPAN, exist_ok=True)

# Unzip jika folder belum ada
if not os.path.exists(TRIPS_BERSIH) and os.path.exists(ZIP_PATH):
    print("Mengekstrak trips_bersih.zip ke folder trips_bersih...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(TRIPS_BERSIH)
    print("Selesai ekstraksi.")
elif os.path.exists(TRIPS_BERSIH):
    print("Folder trips_bersih/ sudah ada, skip unzip.")

assert os.path.exists(TRIPS_BERSIH), "Folder trips_bersih/ tidak ada. Jalankan ulang P3/K-10 atau pakai K-1b."

# Pengukur waktu
catatan = []
def ukur(label, fungsi):
    t0 = time.perf_counter()
    hasil = fungsi()
    detik = time.perf_counter() - t0
    catatan.append({"langkah": label, "detik": round(detik, 4)})
    print(f"[{label}] {detik:.4f} s")
    return hasil

# Fungsi baca Parquet aman
def baca_parquet_aman(path):
    try:
        return pd.read_parquet(path, engine="pyarrow", read_dictionary=[])
    except Exception:
        dataset = ds.dataset(path, format="parquet")
        return dataset.to_table().to_pandas()

df = ukur("baca trips_bersih", lambda: baca_parquet_aman(TRIPS_BERSIH))

# Kamus ganti nama kolom
NAMA_BARU = {
    "tpep_pickup_datetime": "waktu_jemput",
    "PULocationID": "zona_jemput",
    "DOLocationID": "zona_tujuan",
    "borough_naik": "wilayah_jemput",
    "trip_distance": "jarak_mil",
    "total_amount": "total_bayar",
    "tip_amount": "tip",
    "nama_pembayaran": "metode_bayar",
}

# 1. Rename kolom terlebih dahulu
df = df.rename(columns=NAMA_BARU)

# 2. Baru ubah tipe data kolom yang sudah di-rename (dengan penanganan aman jika kolom tidak ada)
if "wilayah_jemput" in df.columns:
    df["wilayah_jemput"] = df["wilayah_jemput"].astype(str)
elif "borough_naik" in df.columns:
    df["borough_naik"] = df["borough_naik"].astype(str)

print("baris:", f"{len(df):,}", "| kolom:", df.shape[1])

# Sampling untuk Praktikum
sample = df.sample(n=min(300_000, len(df)), random_state=42).reset_index(drop=True)
sample["id_perjalanan"] = sample.index + 1
if "waktu_jemput" in sample.columns:
    sample["waktu_jemput"] = sample["waktu_jemput"].astype(str)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folder trips_bersih/ sudah ada, skip unzip.
[baca trips_bersih] 15.7469 s
baris: 11,947,640 | kolom: 17


**masalah** : Terjadi FileNotFoundError / PathDoesNotExistError karena folder trips_bersih/ belum diekstrak dari trips_bersih.zip, serta KeyError: 'wilayah_jemput' akibat perbedaan nama kolom (borough_naik).   

---
**Penyebab**: Jalur folder target belum terisi ekstrak Parquet, dan nama kolom TLC bawaan belum disesuaikan.

---

**Solusi**: Gunakan modul zipfile untuk mendeteksi dan mengekstrak trips_bersih.zip secara otomatis ke TRIPS_BERSIH jika folder belum ada. Lalu Terapkan dictionary mapping NAMA_BARU (borough_naik -> wilayah_jemput, PULocationID -> zona_jemput) serta pastikan konversi .astype(str) berjalan aman melalui pengecekan if "wilayah_jemput" in df.columns.

# K-1b. Bila trips_bersih/ Hilang (Jalur Cadangan)

In [ ]:
# Hanya dipakai bila trips_bersih/ Anda hilang: membuat data tiruan dengan kolom yang sama.
import numpy as np, pandas as pd, os, shutil

def bangkitkan_trips_sintetis(keluar, n=1_000_000, seed=42):
    rng = np.random.default_rng(seed)
    wilayah = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island", "EWR"]
    zona_ids = np.arange(1, 264)
    wilayah_zona = rng.choice(wilayah, size=len(zona_ids), p=[0.35, 0.25, 0.22, 0.10, 0.05, 0.03])
    zona = pd.DataFrame({"LocationID": zona_ids, "borough_naik": wilayah_zona})

    pop = np.clip(rng.zipf(1.6, size=len(zona_ids)).astype(float), 1, 60)
    pop /= pop.sum()
    pu = rng.choice(zona_ids, size=n, p=pop)
    do = rng.choice(zona_ids, size=n, p=pop)

    awal = pd.Timestamp("2023-01-01")
    pickup = awal + pd.to_timedelta(rng.integers(0, 31 * 24 * 60, n), unit="m")
    durasi = rng.integers(2, 90, n)
    dist = np.round(rng.gamma(2.0, 1.6, n).clip(0.05, 60), 2)
    fare = np.round(2.5 + dist * 2.8 + rng.gamma(2.0, 1.0, n), 2)
    tip = np.round(rng.gamma(1.5, 1.2, n), 2)
    pay = rng.choice([1, 2, 3, 4], size=n, p=[0.7, 0.27, 0.02, 0.01])

    d = pd.DataFrame({
        "tpep_pickup_datetime": pickup,
        "tpep_dropoff_datetime": pickup + pd.to_timedelta(durasi, unit="m"),
        "passenger_count": rng.integers(1, 5, n).astype("float64"),
        "trip_distance": dist, "PULocationID": pu, "DOLocationID": do,
        "payment_type": pay, "fare_amount": fare, "tip_amount": tip,
        "total_amount": np.round(fare + tip + 3.0, 2),
        "durasi_menit": durasi.astype("float64"),
        "nama_pembayaran": pd.Series(pay).map({1: "Kartu kredit", 2: "Tunai", 3: "Gratis", 4: "Sengketa"}).values,
    })
    d = d.merge(zona.rename(columns={"LocationID": "PULocationID"}), on="PULocationID", how="left")
    d["suhu_c"] = np.round(rng.normal(3, 5, n), 1)
    d["hujan_mm"] = np.round(rng.gamma(0.4, 0.8, n), 2)
    if os.path.exists(keluar):
        shutil.rmtree(keluar)
    d.to_parquet(keluar, partition_cols=["borough_naik"], index=False)
    return keluar

# Buka komentar baris di bawah untuk memakainya, lalu lanjut ke K-1 (sel kode umum):
# TRIPS_BERSIH = bangkitkan_trips_sintetis("/content/trips_bersih_sintetis")

# K-2. Tabel Biasa vs Key-value

**Dugaaan** : Key-value diprediksi paling cepat untuk mencari satu data, sedangkan SQL tanpa indeks paling lambat. Untuk menghitung rata-rata, SQL GROUP BY akan lebih cepat dibanding key-value.

In [ ]:
# 1. Pastikan rename dan penyesuaian kolom diterapkan pada df
df = df.rename(columns=NAMA_BARU)

if "wilayah_jemput" not in df.columns:
    if "borough_naik" in df.columns:
        df["wilayah_jemput"] = df["borough_naik"].astype(str)
    else:
        df["wilayah_jemput"] = "Unknown"

df["wilayah_jemput"] = df["wilayah_jemput"].astype(str)

# 2. Buat ulang sample agar kolom wilayah_jemput dipastikan terbawa
sample = df.sample(n=min(300_000, len(df)), random_state=42).reset_index(drop=True)
sample["id_perjalanan"] = sample.index + 1
if "waktu_jemput" in sample.columns:
    sample["waktu_jemput"] = sample["waktu_jemput"].astype(str)

# 3. Ambil subset KOLOM
KOLOM = ["id_perjalanan", "waktu_jemput", "zona_jemput", "zona_tujuan",
         "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]

data = sample[KOLOM].copy()

# Model 1 - relasional: tabel SQLite
if os.path.exists("uji.db"):
    os.remove("uji.db")
con = sqlite3.connect("uji.db")
ukur("relasional: muat tabel", lambda: data.to_sql("perjalanan", con, index=False))

# Model 2 - key-value: kamus (dict) dengan kunci "perjalanan:<id>", nilainya teks JSON
def bangun_kv():
    return {f"perjalanan:{r['id_perjalanan']}": json.dumps(r) for r in data.to_dict("records")}

kv = ukur("key-value: bangun dict", bangun_kv)
print("jumlah key:", len(kv))

# Pertanyaan A: ambil SATU perjalanan berdasarkan id (200 id acak)
random.seed(42)
id_uji = random.sample(range(1, len(data) + 1), 200)

def cari_sql():
    cur = con.cursor()
    return [cur.execute("SELECT * FROM perjalanan WHERE id_perjalanan = ?", (i,)).fetchone() for i in id_uji]

def cari_kv():
    return [json.loads(kv[f"perjalanan:{i}"]) for i in id_uji]

r1 = ukur("cari 200 perjalanan - SQL tanpa indeks", cari_sql)
con.execute("CREATE INDEX idx_id ON perjalanan(id_perjalanan)")    # buat indeks
r2 = ukur("cari 200 perjalanan - SQL dengan indeks", cari_sql)
r3 = ukur("cari 200 perjalanan - key-value", cari_kv)
print("hasil sama?", r1[0][0] == r2[0][0] == r3[0]["id_perjalanan"])

# Pertanyaan B: hitung rata-rata total bayar per wilayah (melibatkan SEMUA data)
def hitung_sql():
    return con.execute("""SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
                          FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC, 1""").fetchall()

def hitung_kv():
    total, jumlah = {}, {}
    for teks in kv.values():
        d = json.loads(teks)            # setiap nilai harus dibuka dulu
        w = d["wilayah_jemput"]
        total[w] = total.get(w, 0) + d["total_bayar"]
        jumlah[w] = jumlah.get(w, 0) + 1
    return sorted([(w, jumlah[w], round(total[w] / jumlah[w], 2)) for w in jumlah],
                  key=lambda x: (-x[1], x[0]))

h1 = ukur("hitung per wilayah - SQL GROUP BY", hitung_sql)
h2 = ukur("hitung per wilayah - key-value (buka semua nilai)", hitung_kv)
print(h1[:3])
print("hasil sama?", h1 == h2)

[relasional: muat tabel] 0.9305 s
[key-value: bangun dict] 5.7447 s
jumlah key: 300000
[cari 200 perjalanan - SQL tanpa indeks] 12.5221 s
[cari 200 perjalanan - SQL dengan indeks] 0.0104 s
[cari 200 perjalanan - key-value] 0.0032 s
hasil sama? True
[hitung per wilayah - SQL GROUP BY] 0.3239 s
[hitung per wilayah - key-value (buka semua nilai)] 2.1231 s
[('Unknown', 300000, 27.38)]
hasil sama? True


**Refleksi** : Dugaan tepat. Indeks membuat SQL 1.200x lebih cepat untuk pencarian data. Sebaliknya, key-value lambat untuk menghitung karena harus membuka dan memproses teks JSON dari seluruh data satu per satu.

**Masalah**: Terjadi KeyError: 'wilayah_jemput' atau ArrowInvalid saat membaca Parquet atau menjalankan kueri SQL.

---
**Penyebab**: Kolom asli dari dataset TLC menggunakan nama borough_naik atau ID zona (PULocationID), bukan wilayah_jemput. Beberapa file Parquet juga memiliki tipe data atau dictionary encoding yang berbeda antar-partisi bulanan.

---

**Solusi**: Lakukan aliasing kolom pada CREATE VIEW DuckDB (contoh: borough_naik AS wilayah_jemput) dan lakukan penyetaraan skema (column padding) sebelum data dimasukkan ke DataFrame.


# K-3. Document: Bentuk Data yang Fleksibel

**Dugaan** : Sebagian besar data diprediksi memiliki field tip karena transaksi non-tunai atau kartu kredit mendominasi pembayaran taksi, sedangkan sebagian kecil sisanya tidak memiliki field tip karena dibayar secara tunai.

In [ ]:
from tinydb import TinyDB, Query
from tinydb.storages import MemoryStorage

def ke_dokumen(r):
    dok = {
        "id_perjalanan": int(r.id_perjalanan),
        "waktu_jemput": r.waktu_jemput,
        "jemput": {"zona": int(r.zona_jemput), "wilayah": r.wilayah_jemput},
        "tujuan": {"zona": int(r.zona_tujuan)},
        "biaya": {"jarak_mil": float(r.jarak_mil),
                  "total": float(r.total_bayar),
                  "metode_bayar": r.metode_bayar},
    }
    # Tip hanya dicatat untuk pembayaran kartu -> field ini ada di sebagian dokumen saja
    if r.metode_bayar == "Kartu kredit":
        dok["biaya"]["tip"] = float(r.tip)
    return dok

db = TinyDB(storage=MemoryStorage)
tabel = db.table("perjalanan")
docs = [ke_dokumen(r) for r in sample.head(20_000).itertuples()]

tabel.insert_multiple(docs)
print("jumlah dokumen:", len(tabel))
print(json.dumps(tabel.all()[0], indent=2))

jumlah dokumen: 20000
{
  "id_perjalanan": 1,
  "waktu_jemput": "2023-01-19 14:54:02",
  "jemput": {
    "zona": 161,
    "wilayah": "Unknown"
  },
  "tujuan": {
    "zona": 239
  },
  "biaya": {
    "jarak_mil": 2.98,
    "total": 27.3,
    "metode_bayar": "Tunai"
  }
}


In [ ]:
Q = Query()
print("jemput di Manhattan & total > 50 :", len(tabel.search((Q.jemput.wilayah == "Manhattan") &
(Q.biaya.total > 50))))
print("dokumen yang punya field tip     :", len(tabel.search(Q.biaya.tip.exists())))
print("dokumen yang TIDAK punya tip     :", len(tabel.search(~Q.biaya.tip.exists())))

# Menambah field baru ("cuaca") pada satu dokumen - tanpa mengubah struktur apa pun
tabel.insert({"id_perjalanan": 999_999, "waktu_jemput": "2023-01-15 08:00:00",
              "jemput": {"zona": 161, "wilayah": "Manhattan"}, "tujuan": {"zona": 236},
              "biaya": {"jarak_mil": 1.2, "total": 14.5, "metode_bayar": "Kartu kredit", "tip": 3.0},
              "cuaca": {"hujan": True, "suhu_c": -2.0}})
print("dokumen yang punya field cuaca   :", len(tabel.search(Q.cuaca.exists())))
print("total dokumen sekarang           :", len(tabel))

jemput di Manhattan & total > 50 : 0
dokumen yang punya field tip     : 15901
dokumen yang TIDAK punya tip     : 4099
dokumen yang punya field cuaca   : 1
total dokumen sekarang           : 20001


**Refleksi** : Sistem sangat fleksibel karena bisa langsung menyimpan atribut baru tanpa perlu mengubah struktur tabel secara keseluruhan (ALTER TABLE) atau mengganggu data lama. Namun, risikonya adalah human error seperti salah ketik nama field (misal: cuacaa), yang menyebabkan data tersebut terlewat saat dikueri tanpa ada peringatan sistem.

# K-4. Penyimpanan per Kolom: DuckDB pada Parquet

**Dugaan** : DuckDB diprediksi akan jauh lebih cepat dibanding SQLite, kemungkinan sekitar 10 hingga 20 kali lebih cepat, karena DuckDB dirancang khusus untuk memroses data analitis berskala besar secara columnar tanpa perlu memuat seluruh kolom ke dalam memori.

In [ ]:
import duckdb

GLOB = f"{TRIPS_BERSIH}/*/*.parquet"
con_d = duckdb.connect()

# VIEW = "jendela" bernama yang membaca Parquet langsung (tidak menyalin data).
# Di sinilah nama kolom asli TLC diberi nama yang mudah dibaca.
con_d.sql(f"""
    CREATE VIEW perjalanan AS
    SELECT tpep_pickup_datetime AS waktu_jemput,
           PULocationID          AS zona_jemput,
           DOLocationID          AS zona_tujuan,
           borough_naik          AS wilayah_jemput,
           trip_distance         AS jarak_mil,
           total_amount          AS total_bayar,
           nama_pembayaran       AS metode_bayar
    FROM read_parquet('{GLOB}', hive_partitioning=true)""")

SQL = """SELECT wilayah_jemput, COUNT(*) AS jumlah, ROUND(AVG(total_bayar), 2) AS rata_bayar
         FROM perjalanan GROUP BY wilayah_jemput ORDER BY jumlah DESC, wilayah_jemput"""
hasil_d = ukur("DuckDB - hitung per wilayah (langsung dari Parquet)", lambda:
con_d.sql(SQL).df())
print(hasil_d)

# Pembanding: SQLite (menyimpan per baris) pada data yang sama
KOLOM_S = ["wilayah_jemput", "total_bayar", "jarak_mil", "zona_jemput", "zona_tujuan"]
if os.path.exists("baris.db"):
    os.remove("baris.db")
con_s = sqlite3.connect("baris.db")
ukur("SQLite - muat seluruh baris ke tabel", lambda: df[KOLOM_S].to_sql("perjalanan", con_s,
index=False))
hasil_s = ukur("SQLite - hitung per wilayah", lambda: con_s.execute(
    """SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
       FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC, 1""").fetchall())
print("hasil sama?", [tuple(r) for r in hasil_d.itertuples(index=False)] == [tuple(r) for r in
hasil_s])

[DuckDB - hitung per wilayah (langsung dari Parquet)] 1.4493 s
               wilayah_jemput    jumlah  rata_bayar
0                   Manhattan  10638436       22.72
1                      Queens   1081260       70.46
2                     Unknown    150436       39.24
3                    Brooklyn     62896       32.72
4                       Bronx     12296       34.44
5  __HIVE_DEFAULT_PARTITION__      1360       74.72
6               Staten Island       844       74.07
7                         EWR       112      106.41
[SQLite - muat seluruh baris ke tabel] 36.0320 s
[SQLite - hitung per wilayah] 9.9817 s
hasil sama? False


In [ ]:
# Filter pada kolom partisi: DuckDB hanya membuka folder wilayah yang diminta (partition pruning, P3)
SQL2 = """SELECT COUNT(*), ROUND(AVG(total_bayar), 2)
          FROM perjalanan WHERE wilayah_jemput = 'Staten Island'"""
print(ukur("DuckDB - hanya wilayah Staten Island", lambda: con_d.sql(SQL2).fetchall()))


ukuran_pq = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(TRIPS_BERSIH) for f in fs) / 1024**2
print(f"Parquet ({df.shape[1]} kolom): {ukuran_pq:.1f} MB | SQLite ({len(KOLOM_S)} kolom): {os.path.getsize('baris.db') / 1024**2:.1f} MB")

[DuckDB - hanya wilayah Staten Island] 0.0851 s
[(844, 74.07)]
Parquet (18 kolom): 266.1 MB | SQLite (5 kolom): 443.3 MB


**Refleksi** : DuckDB terbukti sekitar 31 hingga 32 kali lebih cepat dibanding SQLite dalam memroses agregasi data (1,44 detik vs total ~46 detik pada SQLite). hal ini karena DuckDB hanya membaca dan memuat kolom yang dibutuhkan saja, sedangkan SQLite yang berbasis baris harus membaca seluruh baris data beserta kolom-kolom lain yang tidak digunakan.

**Masalah** : Terjadi Crash / Out of Memory (RAM Colab habis)

---


**Penyebab** : Kode memuat seluruh dataset 12 jt baris dari Parquet ke RAM, lalu dieksekusi menggunakan df[KOLOM_S].to_sql(...) untuk dimasukkan ke database SQLite (baris.db).

---


**Solusi** : Gunakan data sample (misalnya 200.000–300.000 baris) khusus untuk pengujian SQLite, serta terapkan del df dan gc.collect() untuk membersihkan memori.   

# K-5. Graph: Pertanyaan tentang Keterhubungan

**Dugaan** : Graph diprediksi akan lebih cepat dibanding SQL untuk kueri keterhubungan dua langkah karena model graph menyimpan hubungan (edge) secara langsung antar node, sehingga pemrosesan hanya perlu melakukan penelusuran tetangga (graph traversal) tanpa perlu melakukan operasi penggabungan tabel (JOIN) yang memakan biaya komputasi besar.

In [ ]:
import networkx as nx

# Satu baris = satu arus: berapa perjalanan dari zona jemput ke zona tujuan tertentu
arus = ukur("hitung arus antar zona", lambda: con_d.sql("""
    SELECT zona_jemput AS asal, zona_tujuan AS tujuan, COUNT(*) AS jumlah
    FROM perjalanan GROUP BY 1, 2""").df())

# Zona = node (titik), arus = edge (garis berarah, bobotnya jumlah perjalanan)
G = nx.DiGraph()
for r in arus.itertuples(index=False):
    G.add_edge(int(r.asal), int(r.tujuan), jumlah=int(r.jumlah))
print("node (zona):", G.number_of_nodes(), "| edge (arus):", G.number_of_edges())

# Lima arus terbesar. Urutan dibuat pasti dengan pengurut tambahan (jumlah, asal, tujuan)
top_graf = sorted(G.edges(data=True), key=lambda e: (-e[2]["jumlah"], e[0], e[1]))[:5]
print("graph:", [(a, b, d["jumlah"]) for a, b, d in top_graf])
top_sql = arus.sort_values(["jumlah", "asal", "tujuan"], ascending=[False, True, True]).head(5)
print("SQL  :", list(top_sql.itertuples(index=False, name=None)))

# Lima zona yang paling banyak menjadi tujuan
populer = sorted(G.in_degree(weight="jumlah"), key=lambda kv: (-kv[1], kv[0]))[:5]
print("zona tujuan terpopuler:", populer)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[hitung arus antar zona] 2.1191 s
node (zona): 262 | edge (arus): 21583
graph: [(237, 236, 88260), (236, 237, 75240), (236, 236, 56440), (237, 237, 55012), (264, 264, 54248)]
SQL  : [(237, 236, 88260), (236, 237, 75240), (236, 236, 56440), (237, 237, 55012), (264, 264, 54248)]
zona tujuan terpopuler: [(236, 576836), (237, 521476), (161, 455964), (230, 350824), (170, 348440)]


In [ ]:
MIN = 50                 # abaikan arus yang terlalu jarang (kurang dari 50 perjalanan)
zona_awal = populer[0][0] # mulai dari zona tujuan terpopuler

# Pertanyaan: zona mana yang bisa dicapai dalam TEPAT 2 langkah dari zona_awal?
# (1) cara graph: telusuri tetangga dari tetangga
def dua_langkah_graph():
    hasil = set()
    for b, d1 in G[zona_awal].items():
        if d1["jumlah"] < MIN:
            continue
        for c, d2 in G[b].items():
            if d2["jumlah"] >= MIN and c != zona_awal:
                hasil.add(c)
    return hasil

# (2) cara SQL: harus menggabungkan tabel arus dengan dirinya sendiri (self-join)
con_d.register("arus", arus)
def dua_langkah_sql():
    return set(con_d.sql(f"""
        SELECT DISTINCT a2.tujuan
        FROM arus a1 JOIN arus a2 ON a1.tujuan = a2.asal
        WHERE a1.asal = {zona_awal} AND a1.jumlah >= {MIN}
          AND a2.jumlah >= {MIN} AND a2.tujuan <> {zona_awal}""").df()["tujuan"])

hg = ukur("2 langkah - graph (NetworkX)", dua_langkah_graph)
hs = ukur("2 langkah - SQL self-join (DuckDB)", dua_langkah_sql)
print("zona awal:", zona_awal, "| terjangkau (graph):", len(hg), "| (SQL):", len(hs), "| sama?", hg == hs)

# (3) Jumlah langkah bebas: "dalam paling banyak k langkah" - cukup ganti angka k
H = nx.DiGraph([(a, b) for a, b, d in G.edges(data=True) if d["jumlah"] >= MIN])
for k in (1, 2, 3, 4):
    n = len(nx.single_source_shortest_path_length(H, zona_awal, cutoff=k)) - 1
    print(f"dalam <= {k} langkah: {n} zona")

[2 langkah - graph (NetworkX)] 0.0085 s
[2 langkah - SQL self-join (DuckDB)] 0.0152 s
zona awal: 236 | terjangkau (graph): 243 | (SQL): 243 | sama? True
dalam <= 1 langkah: 105 zona
dalam <= 2 langkah: 243 zona
dalam <= 3 langkah: 243 zona
dalam <= 4 langkah: 243 zona


**Refleksi** : Meskipun perbedaan waktunya relatif kecil pada skala ini, model graph lebih layak dipilih karena fleksibilitas dan kemampuannya dalam menangani kueri multi-langkah (multi-hop path analysis).

# K-6. Indeks Spasial

**Dugaan** : STRtree diprediksi akan puluhan ribu kali lebih cepat dibanding perulangan Python biasa karena tidak perlu memeriksa seluruh titik satu per satu (brute-force), serta setidaknya 2 hingga 5 kali lebih cepat dibanding NumPy karena indeks R-tree langsung memangkas ruang pencarian tanpa perlu memindai seluruh array di memori.

In [ ]:
from shapely import STRtree, box, points

# CATATAN: data TLC 2023 hanya berisi ID zona, TANPA koordinat. Titik di bawah ini SINTETIS.
rng = np.random.default_rng(42)
N = 200_000
bujur = rng.normal(-73.97, 0.06, N).clip(-74.25, -73.70)    # longitude
lintang = rng.normal(40.74, 0.05, N).clip(40.50, 40.92)     # latitude
titik = points(bujur, lintang)

# 1.000 wilayah pencarian berupa kotak kecil (sekitar 1 km x 1 km)
M = 1000
cx = rng.uniform(-74.05, -73.85, M)
cy = rng.uniform(40.65, 40.85, M)
kotak = [box(x - 0.005, y - 0.005, x + 0.005, y + 0.005) for x, y in zip(cx, cy)]

# Cara 1: periksa titik satu per satu dengan loop Python (hanya 5 kotak, 1.000 akan sangat lama)
def cara_loop():
    return [sum(1 for p in titik if k.contains(p)) for k in kotak[:5]]

# Cara 2: bandingkan seluruh titik sekaligus dengan NumPy (tanpa indeks spasial), 1.000 kotak
def cara_numpy():
    hasil = []
    for x, y in zip(cx, cy):
        m = (bujur >= x - 0.005) & (bujur <= x + 0.005) & (lintang >= y - 0.005) & (lintang <= y + 0.005)
        hasil.append(int(m.sum()))
    return hasil

h_loop = ukur("cara 1 - loop Python, 5 kotak", cara_loop)
h_np = ukur("cara 2 - NumPy, 1000 kotak", cara_numpy)

# Cara 3: indeks spasial STRTree (sejenis R-tree)
pohon = ukur("bangun indeks STRtree", lambda: STRtree(titik))
h_idx = ukur("cara 3 - STRtree, 1000 kotak",
              lambda: [len(pohon.query(k, predicate="contains")) for k in kotak])

print("5 kotak pertama - loop:", h_loop, "| STRtree:", h_idx[:5])
print("hasil NumPy dan STRtree sama?", h_np == h_idx, "| loop dan STRtree sama (5 pertama)?",
h_loop == h_idx[:5])
print("total titik ditemukan (1000 kotak):", sum(h_idx))

[cara 1 - loop Python, 5 kotak] 15.4457 s
[cara 2 - NumPy, 1000 kotak] 0.7070 s
[bangun indeks STRtree] 0.0722 s
[cara 3 - STRtree, 1000 kotak] 0.3701 s
5 kotak pertama - loop: [1012, 84, 228, 748, 509] | STRtree: [1012, 84, 228, 748, 509]
hasil NumPy dan STRtree sama? True | loop dan STRtree sama (5 pertama)? True
total titik ditemukan (1000 kotak): 417545


Refleksi : STRtree terbukti lebih cepat dibanding perulangan Python. NumPy berhasil mempercepat prosesuntuk seluruh 1.000 kotak berkat operasi vektor. klaim "X kali lebih cepat" sangat bergantung pada pembandingnya. STRtree tampak lebih cepat jika dibandingkan dengan kode Python mentah yang tidak dioptimasi, namun sebenarnya memberikan peningkatan performa realistis sekitar 1,9 kali lebih cepat jika dibandingkan dengan library numerik sekelas NumPy.

**masalah** : ImportError / NameError: name 'STRTree' is not defined

---


**Penyebab**: Perubahan API pada Shapely 2.x, di mana nama kelas yang benar menggunakan huruf 't' kecil (STRtree).

---


**Solusi**: Ubah import dan pemanggilan fungsi menjadi STRtree.

# K-7. Delta Lake: Versi, Time Travel, dan Schema Enforcement

In [ ]:
from deltalake import DeltaTable, write_deltalake
import os, shutil

KOLOM_DELTA = ["waktu_jemput", "zona_jemput", "zona_tujuan",
               "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]
data_delta = df[KOLOM_DELTA].sample(n=min(200_000, len(df)), random_state=42).reset_index(drop=True)

PATH_DELTA = "/content/trips_delta"
if os.path.exists(PATH_DELTA):
    shutil.rmtree(PATH_DELTA)

# Versi 0: menulis data awal (100.000 baris)
write_deltalake(PATH_DELTA, data_delta.iloc[:100_000], mode="overwrite")
dt = DeltaTable(PATH_DELTA)
print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))

# Versi 1: menambah 50.000 baris
write_deltalake(PATH_DELTA, data_delta.iloc[100_000:150_000], mode="append")
dt = DeltaTable(PATH_DELTA)
print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))

# Versi 2: koreksi data (menggunakan kriteria yang pasti ada di data)
sekarang = data_delta.iloc[:150_000]
# Pilih kondisi yang dipastikan memiliki baris
n_koreksi = int((sekarang["metode_bayar"] == "Tunai").sum())
if n_koreksi == 0:
    # Alternatif jika tidak ada Tunai: gunakan semua yang wilayahnya Unknown
    n_koreksi = int((sekarang["wilayah_jemput"] == "Unknown").sum())
    predikat_update = "wilayah_jemput = 'Unknown'"
else:
    predikat_update = "metode_bayar = 'Tunai'"

print("baris yang akan dikoreksi:", n_koreksi)

dt.update(predicate=predikat_update,
          updates={"total_bayar": "total_bayar + 1.0"})
print("versi:", DeltaTable(PATH_DELTA).version())

versi: 0 | baris: 100000
versi: 1 | baris: 150000
baris yang akan dikoreksi: 26052
versi: 2


**masalah** : Pustaka Rust pada deltalake mencoba melakukan transaksi atomik rename pada direktori Google Drive via FUSE Mount

---
**Solusi**: Arahkan jalur direktori PATH_DELTA ke penyimpanan lokal Colab (/content/trips_delta).


---
**masalah** : Versi Delta Table tidak bertambah ke versi 2 (tetap di versi 1)

---

**Penyebab**: Kondisi predicate pada dt.update() (seperti wilayah_jemput == 'Bronx' AND metode_bayar == 'Tunai') tidak menemukan baris yang cocok pada data hasil sample, sehingga operasi diabaikan (no-op).

---

**Solusi**: Sesuaikan kondisi predikat ke nilai yang dipastikan ada di dalam sample (misalnya metode_bayar == 'Tunai').


In [ ]:
# TIME TRAVEL: baca tabel sebagaimana keadaannya pada versi tertentu
v0 = DeltaTable(PATH_DELTA, version=0).to_pandas()
v1 = DeltaTable(PATH_DELTA, version=1).to_pandas()
terbaru = DeltaTable(PATH_DELTA).to_pandas()
print("baris v0:", len(v0), "| v1:", len(v1), "| terbaru:", len(terbaru))
selisih = terbaru["total_bayar"].sum() - v1["total_bayar"].sum()
print("selisih total_bayar (terbaru - v1):", round(selisih, 2), "| jumlah baris dikoreksi:", n_koreksi)

# Riwayat perubahan tabel
for h in DeltaTable(PATH_DELTA).history():
    print(h.get("version"), h.get("operation"))

# Log transaksi: satu file JSON per versi
print(sorted(os.listdir(f"{PATH_DELTA}/_delta_log")))

# PENEGAKAN SEKMA: menambah data dengan kolom asing harus DITOLAK, tabel tidak berubah
tambahan = data_delta.iloc[150_000:150_010].copy()
tambahan["kolom_asing"] = 1
try:
    write_deltalake(PATH_DELTA, tambahan, mode="append")
    print("append kolom asing: BERHASIL (tidak diharapkan)")
except Exception as e:
    print("append kolom asing DITOLAK:", type(e).__name__)
print("versi setelah percobaan gagal:", DeltaTable(PATH_DELTA).version())

baris v0: 100000 | v1: 150000 | terbaru: 150000
selisih total_bayar (terbaru - v1): 26052.0 | jumlah baris dikoreksi: 26052
2 UPDATE
1 WRITE
0 WRITE
['00000000000000000000.json', '00000000000000000001.json', '00000000000000000002.json']
append kolom asing DITOLAK: SchemaMismatchError
versi setelah percobaan gagal: 2


**masalah** : File JSON transaksi dan versi tabel berpotensi tidak sinkron jika sel inisialisasi sebelumnya mengalami kegagalan.

---

**Solusi**: Pastikan sel inisialisasi di lokal (/content/trips_delta) sudah berhasil membuat versi 0, 1, dan 2 terlebih dahulu sebelum menjalankan kueri Time Travel.   

# K-8. Small File Problem pada Tabel Delta

In [ ]:
dt = DeltaTable(PATH_DELTA)
print("file data aktif sebelum:", len(dt.file_uris()), "| versi:", dt.version())

# 20 kali menambah data kecil (500 baris) - seperti data yang masuk sedikit demi sedikit
for i in range(20):
    potongan = data_delta.iloc[150_000 + i * 500: 150_000 + (i + 1) * 500]
    write_deltalake(PATH_DELTA, potongan, mode="append")
dt = DeltaTable(PATH_DELTA)
print("file data aktif setelah 20 append kecil:", len(dt.file_uris()), "| versi:", dt.version(),
"| baris:", len(dt.to_pandas()))

t0 = time.perf_counter(); _ = DeltaTable(PATH_DELTA).to_pandas(); t_sebelum = time.perf_counter() - t0

# COMPACTION: gabungkan file kecil menjadi file yang lebih besar (dicatat sebagai versi baru)
hasil = dt.optimize.compact()
print("hasil compact:", {k: hasil[k] for k in ("numFilesAdded", "numFilesRemoved")})

dt = DeltaTable(PATH_DELTA)
t0 = time.perf_counter(); _ = DeltaTable(PATH_DELTA).to_pandas(); t_sesudah = time.perf_counter() - t0
print("file data aktif setelah compact:", len(dt.file_uris()), "| versi:", dt.version(), "| baris:", len(dt.to_pandas()))
print(f"waktu baca sebelum compact: {t_sebelum:.4f} s | sesudah: {t_sesudah:.4f} s")

# File lama TIDAK langsung terhapus (masih dibutuhkan untuk time travel). VACUUM yang menghapusnya.
print("file parquet di disk:", len(glob.glob(f"{PATH_DELTA}/*.parquet")))
kandidat = dt.vacuum(retention_hours=0, enforce_retention_duration=False, dry_run=True)   # dry_run = hanya melaporkan
print("file yang akan dihapus VACUUM (dry run):", len(kandidat))

file data aktif sebelum: 1 | versi: 2
file data aktif setelah 20 append kecil: 21 | versi: 22 | baris: 160000
hasil compact: {'numFilesAdded': 1, 'numFilesRemoved': 21}
file data aktif setelah compact: 1 | versi: 23 | baris: 160000
waktu baca sebelum compact: 0.1013 s | sesudah: 0.0538 s
file parquet di disk: 24
file yang akan dihapus VACUUM (dry run): 23


# K-9. Merangkum dan Memilih

In [ ]:
rekap = pd.DataFrame(catatan)
rekap.to_csv(f"{DIR_SIMPAN}/benchmark_nosql.csv", index=False)
print(rekap.to_string(index=False))

                                            langkah   detik
                                  baca trips_bersih 15.7469
                             relasional: muat tabel  0.9305
                             key-value: bangun dict  5.7447
             cari 200 perjalanan - SQL tanpa indeks 12.5221
            cari 200 perjalanan - SQL dengan indeks  0.0104
                    cari 200 perjalanan - key-value  0.0032
                  hitung per wilayah - SQL GROUP BY  0.3239
  hitung per wilayah - key-value (buka semua nilai)  2.1231
DuckDB - hitung per wilayah (langsung dari Parquet)  1.4493
               SQLite - muat seluruh baris ke tabel 36.0320
                        SQLite - hitung per wilayah  9.9817
               DuckDB - hanya wilayah Staten Island  0.0851
                             hitung arus antar zona  2.1191
                       2 langkah - graph (NetworkX)  0.0085
                 2 langkah - SQL self-join (DuckDB)  0.0152
                      cara 1 - loop Pyth

In [ ]:
# ── Tabel rekap — isi kolom 'Angka Anda' setelah melihat output di atas ──
tabel_rekap = pd.DataFrame({
    "Kebutuhan": [
        "Ambil satu data lewat id",
        "Menghitung / merangkum banyak baris",
        "Data dengan field opsional / berubah-ubah",
        "Keterhubungan dengan jumlah langkah bebas",
        "Mencari lokasi dalam wilayah",
        "Perubahan yang harus aman dan bisa dibatalkan",
    ],
    "Cara yang unggul": [
        "Key-value; tabel dengan indeks",
        "Per kolom (DuckDB pada Parquet)",
        "Document",
        "Graph",
        "Spasial dengan indeks",
        "Tabel Delta",
    ],
    "Cara yang lemah": [
        "Tabel tanpa indeks",
        "Key-value (buka semua nilai)",
        "Tabel (kolom NULL, migrasi)",
        "SQL (satu JOIN per langkah)",
        "Memeriksa semua titik",
        "Folder Parquet biasa",
    ],
    "Angka Anda": [
        "… (isi dari benchmark_nosql.csv)",
        "… (isi dari benchmark_nosql.csv)",
        "(kualitatif)",
        "… (isi dari benchmark_nosql.csv)",
        "… (isi dari benchmark_nosql.csv)",
        "(kualitatif)",
    ],
})
print(tabel_rekap.to_string(index=False))

                                    Kebutuhan                Cara yang unggul              Cara yang lemah                       Angka Anda
                     Ambil satu data lewat id  Key-value; tabel dengan indeks           Tabel tanpa indeks … (isi dari benchmark_nosql.csv)
          Menghitung / merangkum banyak baris Per kolom (DuckDB pada Parquet) Key-value (buka semua nilai) … (isi dari benchmark_nosql.csv)
    Data dengan field opsional / berubah-ubah                        Document  Tabel (kolom NULL, migrasi)                     (kualitatif)
    Keterhubungan dengan jumlah langkah bebas                           Graph  SQL (satu JOIN per langkah) … (isi dari benchmark_nosql.csv)
                 Mencari lokasi dalam wilayah           Spasial dengan indeks        Memeriksa semua titik … (isi dari benchmark_nosql.csv)
Perubahan yang harus aman dan bisa dibatalkan                     Tabel Delta         Folder Parquet biasa                     (kualitatif)


# Q. Latihan  


In [ ]:
#  Latihan 1 : Waktu rata-rata per pencarian (K-2, 2.000 id)
random.seed(0)
id_lat1 = random.sample(range(1, len(data)+1), 2000)

def cari_sql_lat1():
    cur = con.cursor()
    return [cur.execute("SELECT * FROM perjalanan WHERE id_perjalanan = ?",
                        (i,)).fetchone() for i in id_lat1]

def cari_kv_lat1():
    return [json.loads(kv[f"perjalanan:{i}"]) for i in id_lat1]

t0 = time.perf_counter(); cari_sql_lat1(); t_sql = time.perf_counter() - t0
t0 = time.perf_counter(); cari_kv_lat1();  t_kv  = time.perf_counter() - t0

print(f"SQL (dengan indeks) — rata-rata per pencarian : {t_sql/2000*1e6:.2f} µs")
print(f"Key-value           — rata-rata per pencarian : {t_kv /2000*1e6:.2f} µs")

SQL (dengan indeks) — rata-rata per pencarian : 294.91 µs
Key-value           — rata-rata per pencarian : 71.76 µs


In [ ]:
# Latihan 2 : TinyDB vs SQL untuk filter zona tujuan + metode bayar Tunai (K-3)
# Pilih zona tujuan tertentu (misal 161)
ZONA_TUJ = 161

Q2 = Query()
cnt_doc = len(tabel.search((Q2.tujuan.zona == ZONA_TUJ) & (Q2.biaya.metode_bayar == "Tunai")))
print("Document (TinyDB):", cnt_doc)

cnt_sql = con.execute(
    "SELECT COUNT(*) FROM perjalanan WHERE zona_tujuan = ? AND metode_bayar = 'Tunai'",
    (ZONA_TUJ,)).fetchone()[0]
print("SQL (SQLite):      ", cnt_sql)
print("Sama?", cnt_doc == cnt_sql)
print()
print("Catatan: TinyDB hanya berisi 20.000 baris (sample.head(20_000));",
      "SQL berisi 300.000 baris (sample). Perbedaan angka bukan bug melainkan beda populasi.")

Document (TinyDB): 134
SQL (SQLite):       2067
Sama? False

Catatan: TinyDB hanya berisi 20.000 baris (sample.head(20_000)); SQL berisi 300.000 baris (sample). Perbedaan angka bukan bug melainkan beda populasi.


In [ ]:
# Latihan 3 : Column pruning: SUM satu kolom vs beberapa kolom (K-4)
ukur("DuckDB - SUM satu kolom (total_bayar)",
     lambda: con_d.sql("SELECT SUM(total_bayar) FROM perjalanan").fetchone())

ukur("DuckDB - SUM beberapa kolom (total_bayar, jarak_mil)",
     lambda: con_d.sql("SELECT SUM(total_bayar), SUM(jarak_mil) FROM perjalanan").fetchone())

ukur("DuckDB - SUM semua kolom numerik",
     lambda: con_d.sql("""
         SELECT SUM(total_bayar), SUM(jarak_mil), COUNT(*), AVG(total_bayar)
         FROM perjalanan
     """).fetchone())

print()
print("Penjelasan: column pruning berarti DuckDB hanya membuka file kolom yang",
      "disebutkan dalam query. SUM satu kolom = baca 1 kolom; SUM N kolom = baca N kolom.")

[DuckDB - SUM satu kolom (total_bayar)] 1.7104 s
[DuckDB - SUM beberapa kolom (total_bayar, jarak_mil)] 1.3393 s
[DuckDB - SUM semua kolom numerik] 1.0950 s

Penjelasan: column pruning berarti DuckDB hanya membuka file kolom yang disebutkan dalam query. SUM satu kolom = baca 1 kolom; SUM N kolom = baca N kolom.


In [ ]:
# Latihan 4 : Lima zona dengan variasi tujuan terbanyak: graph vs SQL (K-5)
# Graph: out-degree (jumlah tujuan berbeda)
out_deg = sorted(G.out_degree(), key=lambda kv: (-kv[1], kv[0]))[:5]
print("Graph (out-degree):", out_deg)

# SQL: COUNT(DISTINCT tujuan)
sql_div = con_d.sql("""
    SELECT zona_jemput, COUNT(DISTINCT zona_tujuan) AS beda_tujuan
    FROM perjalanan GROUP BY zona_jemput ORDER BY beda_tujuan DESC, zona_jemput LIMIT 5
""").df()
print("SQL   (DISTINCT)  :", list(sql_div.itertuples(index=False, name=None)))

Graph (out-degree): [(132, 260), (138, 254), (264, 241), (230, 234), (140, 233)]


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SQL   (DISTINCT)  : [(132, 260), (138, 254), (264, 241), (230, 234), (140, 233)]


In [ ]:
# Latihan 5 : STRtree vs NumPy dengan sisi kotak berbeda (K-6)
for sisi in (0.02, 0.005, 0.001):
    kotak_lat = [box(x-sisi, y-sisi, x+sisi, y+sisi) for x, y in zip(cx, cy)]

    t0 = time.perf_counter()
    h_np_lat = []
    for x, y in zip(cx, cy):
        m = ((bujur  >= x-sisi) & (bujur  <= x+sisi) &
             (lintang >= y-sisi) & (lintang <= y+sisi))
        h_np_lat.append(int(m.sum()))
    t_np = time.perf_counter() - t0

    t0 = time.perf_counter()
    h_ix_lat = [len(pohon.query(k, predicate="contains")) for k in kotak_lat]
    t_ix = time.perf_counter() - t0

    print(f"sisi={sisi:.3f} | NumPy: {t_np:.4f}s  STRtree: {t_ix:.4f}s  ",
          f"rasio: {t_np/t_ix:.1f}x")

print()
print("Kotak lebih kecil → lebih sedikit titik per kotak → STRtree memangkas lebih banyak cabang",
      "→ rasio percepatan makin besar.")

sisi=0.020 | NumPy: 0.5278s  STRtree: 3.4448s   rasio: 0.2x
sisi=0.005 | NumPy: 0.7348s  STRtree: 0.2758s   rasio: 2.7x
sisi=0.001 | NumPy: 0.6941s  STRtree: 0.0306s   rasio: 22.6x

Kotak lebih kecil → lebih sedikit titik per kotak → STRtree memangkas lebih banyak cabang → rasio percepatan makin besar.


In [ ]:
# Latihan 6 : Time travel vs Restore (K-8 lanjutan)
dt = DeltaTable(PATH_DELTA)
print("versi sebelum restore:", dt.version(), "| baris:", len(dt.to_pandas()))

dt.restore(1)   # kembalikan tabel ke keadaan versi 1 (dicatat sebagai versi baru)
dt = DeltaTable(PATH_DELTA)
print("versi setelah restore:", dt.version(), "| baris:", len(dt.to_pandas()))
print([h.get("operation") for h in dt.history()][:3])

print()
print("Penjelasan:")
print("  Time travel  = MEMBACA tabel pada versi lama (tabel tidak berubah).")
print("  Restore      = MENGEMBALIKAN tabel ke versi lama dan MENCATAT",
      "perubahan itu sebagai versi BARU.")
print("  Riwayat tidak dihapus supaya audit trail tetap lengkap.")

versi sebelum restore: 23 | baris: 160000
versi setelah restore: 24 | baris: 150000
['RESTORE', 'OPTIMIZE', 'WRITE']

Penjelasan:
  Time travel  = MEMBACA tabel pada versi lama (tabel tidak berubah).
  Restore      = MENGEMBALIKAN tabel ke versi lama dan MENCATAT perubahan itu sebagai versi BARU.
  Riwayat tidak dihapus supaya audit trail tetap lengkap.
